# 05 Trayectorias, simultaneidad, Ley de Garantías y proveedores

- El **Concejo**  no cuenta como "sector vinculado al alcalde"; los solapes Alcaldía + Concejo se reportan en su propia categoría.
- Solapes con un contrato en estado `terminado` se separan: la terminación anticipada no actualiza la fecha final y puede fabricar un solape.
- Intervalos inclusivos `[inicio, fin+1)`: contratos consecutivos ya no pierden un día.
- Firmas de **contratación directa durante la Ley de Garantías** 
- Proveedores empresariales con la clasificación (incluye `S.A.S.`, `S.A.`, `E.S.P.`).
- Las tablas con nombres propios quedan en `uso_interno_verificacion/`, nunca en las salidas públicas.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
# Busca la raíz del proyecto (carpeta que contiene funciones/secop_utils.py)
for _p in [RAIZ, *RAIZ.parents][:6]:
    if (_p / "funciones" / "secop_utils.py").exists():
        RAIZ = _p
        break
else:
    raise FileNotFoundError("No se encontró funciones/secop_utils.py; abra el notebook dentro del proyecto")
sys.path.insert(0, str(RAIZ / "funciones"))
import secop_utils as su

VERSION_NB = "05.v2.0"
ETAPA = "05_patrones"
SALIDA = su.carpeta_etapa(RAIZ, ETAPA)
INTERNO = SALIDA / "uso_interno_verificacion"
INTERNO.mkdir(exist_ok=True)
INICIO_OBS = pd.Timestamp("2021-04-01")   # primer mes con uso regular de SECOP II por la Alcaldía
CORTE_EXCL = pd.Timestamp("2026-09-07")
UMBRAL_SOLAPE = 30
man03, r03 = su.abrir_etapa(RAIZ, "03_cps")
man04, r04 = su.abrir_etapa(RAIZ, "04_comparabilidad")
b = su.leer_csv(r03["base_cps"])
ventanas = su.leer_csv(r04["ventanas"])
ipc = su.leer_csv(r04["ipc"])
b["administracion_firma"] = np.select(
    [b["fecha_firma"].ge(INICIO_OBS) & b["fecha_firma"].lt("2024-01-01"), b["fecha_firma"].ge("2024-01-01")],
    su.ADMINS, default="Fuera de observación")
b["fin_excl"] = b["fecha_fin"] + pd.Timedelta(days=1)
print(f"Contratos: {len(b):,}")

Contratos: 37,574


## 1. Recurrencia de personas (Alcaldía central, desde abril de 2021)

In [2]:
cps = b.loc[b["es_central"] & b["apto_persona"] & b["fecha_firma"].ge(INICIO_OBS)].copy()
cps["anio"] = cps["fecha_firma"].dt.year
por_persona = cps.groupby("documento_identidad").agg(
    contratos=("id_contrato", "size"), anios=("anio", "nunique"),
    con_alfonso=("administracion_firma", lambda s: (s == "Alfonso Eljach").any()),
    con_jonathan=("administracion_firma", lambda s: (s == "Jonathan Vásquez").any()))
por_persona["banda"] = pd.cut(por_persona["contratos"], [0, 1, 2, 4, np.inf], labels=["1", "2", "3-4", "5 o más"]).astype(str)
bandas = por_persona["banda"].value_counts().reindex(["1", "2", "3-4", "5 o más"]).rename("personas").reset_index()
bandas["pct_personas"] = 100 * bandas["personas"] / bandas["personas"].sum()
ambos_gobiernos = int((por_persona["con_alfonso"] & por_persona["con_jonathan"]).sum())

cv = su.leer_csv(r04["contratos_ventana"])
principal = cv.loc[cv["ventana"].eq("anio3_ene_ago")].merge(b[["id_contrato", "documento_identidad"]], on="id_contrato")
docs = principal.groupby("administracion")["documento_identidad"].apply(set)
recurrencia = pd.DataFrame([
    {"indicador": "Personas observadas desde abr-2021", "valor": len(por_persona)},
    {"indicador": "Personas con contratos firmados por ambos gobiernos", "valor": ambos_gobiernos},
    {"indicador": "Personas en ambas ventanas principales (ene-ago 2022 y 2026)",
     "valor": len(docs["Alfonso Eljach"] & docs["Jonathan Vásquez"])},
])
display(bandas.round(1)); recurrencia

,banda,personas,pct_personas
0,1,3478,40.1
1,2,1529,17.6
2,3-4,1708,19.7
3,5 o más,1948,22.5


,indicador,valor
0,Personas observadas desde abr-2021,8663
1,Personas con contratos firmados por ambos gobi...,1579
2,Personas en ambas ventanas principales (ene-ag...,549


## 2. Meses cubiertos por persona y año (serie descriptiva, no ranking)
2022–2023 son años 3–4 de Alfonso; 2024–2025 son años 1–2 de Jonathan: etapas distintas del mandato.

In [3]:
GRUPOS_ATRIBUIBLES = ["Administración central", "Sector descentralizado vinculado"]
iv = b.loc[b["apto_intervalo"] & b["grupo_atribucion"].isin(GRUPOS_ATRIBUIBLES) & b["fecha_firma"].ge(INICIO_OBS)].copy()
filas = []
for anio in [2022, 2023, 2024, 2025]:
    admin = "Alfonso Eljach" if anio < 2024 else "Jonathan Vásquez"
    base_anio = iv.loc[iv["administracion_firma"].eq(admin)]   # contratos firmados por el gobierno en ejercicio
    for ambito, datos in [("Alcaldía central", base_anio.loc[base_anio["es_central"]]),
                          ("Alcaldía + descentralizadas (deduplicado)", base_anio)]:
        cob = su.cobertura_por_persona(datos, f"{anio}-01-01", f"{anio + 1}-01-01")
        dias_anio = (pd.Timestamp(f"{anio + 1}-01-01") - pd.Timestamp(f"{anio}-01-01")).days
        filas.append({"anio": anio, "administracion": admin, "ambito": ambito, "personas": len(cob),
                      "mediana_meses": su.mediana(cob["meses_cubiertos"]),
                      "pct_7_meses_o_mas": 100 * cob["meses_cubiertos"].ge(7).mean(),
                      "personas_anio_equivalentes": cob["dias_cubiertos"].sum() / dias_anio})
persona_anio = pd.DataFrame(filas)
persona_anio.round(2)

,anio,administracion,ambito,personas,mediana_meses,pct_7_meses_o_mas,personas_anio_equivalentes
0,2022,Alfonso Eljach,Alcaldía central,3526,5.27,33.64,1661.09
1,2022,Alfonso Eljach,Alcaldía + descentralizadas (deduplicado),3889,5.91,36.02,1865.44
2,2023,Alfonso Eljach,Alcaldía central,3035,5.68,39.47,1541.18
3,2023,Alfonso Eljach,Alcaldía + descentralizadas (deduplicado),3379,6.01,41.67,1754.55
4,2024,Jonathan Vásquez,Alcaldía central,2661,4.04,30.10,1123.65
5,2024,Jonathan Vásquez,Alcaldía + descentralizadas (deduplicado),3043,4.86,32.14,1329.46
6,2025,Jonathan Vásquez,Alcaldía central,3233,4.01,28.49,1346.79
7,2025,Jonathan Vásquez,Alcaldía + descentralizadas (deduplicado),3657,4.01,29.97,1573.27


## 3. Episodios, brechas y tiempo hasta el siguiente episodio (Kaplan Meier con censura)

In [4]:
ivc = b.loc[b["es_central"] & b["apto_intervalo"] & b["fin_excl"].gt(INICIO_OBS)].copy()
ivc["i"] = ivc["fecha_inicio"].clip(lower=INICIO_OBS)
ivc["f"] = ivc["fin_excl"].clip(upper=CORTE_EXCL)
filas = []
for doc, g in ivc.groupby("documento_identidad", sort=True):
    for n, (i, f) in enumerate(su.unir_intervalos(zip(g["i"], g["f"])), 1):
        filas.append({"documento_identidad": doc, "episodio": n, "inicio": i, "fin_excl": f})
episodios = pd.DataFrame(filas)
episodios["siguiente_inicio"] = episodios.groupby("documento_identidad")["inicio"].shift(-1)
episodios["brecha_dias"] = (episodios["siguiente_inicio"] - episodios["fin_excl"]).dt.days


def km(tiempos, eventos, horizontes=(30, 90, 180)):
    d = pd.DataFrame({"t": tiempos, "e": eventos}).sort_values("t")
    s, salida = 1.0, {}
    for t, g in d.groupby("t"):
        en_riesgo = int((d["t"] >= t).sum())
        s *= 1 - g["e"].sum() / en_riesgo
        for h in horizontes:
            if t <= h:
                salida[h] = 1 - s
    return {f"prob_siguiente_{h}d": salida.get(h, 0.0) for h in horizontes}


filas = []
for v in ventanas.itertuples(index=False):
    e = episodios.loc[episodios["fin_excl"].gt(v.inicio) & episodios["fin_excl"].lt(v.fin_excl)]
    e = e.sort_values(["documento_identidad", "episodio"]).drop_duplicates("documento_identidad")  # un episodio por persona
    evento = e["siguiente_inicio"].lt(v.fin_excl)
    t = np.where(evento, e["brecha_dias"], (v.fin_excl - e["fin_excl"]).dt.days)
    brechas = e.loc[evento, "brecha_dias"]
    filas.append({"ventana": v.ventana, "administracion": v.administracion, "personas": len(e),
                  "brecha_mediana_dias": su.mediana(brechas), **km(t, evento.to_numpy())})
continuidad = pd.DataFrame(filas)
continuidad.round(3)

,ventana,administracion,personas,brecha_mediana_dias,prob_siguiente_30d,prob_siguiente_90d,prob_siguiente_180d
0,anio3_ene_ago,Alfonso Eljach,2725,30.0,0.090,0.183,0.335
1,anio3_ene_ago,Jonathan Vásquez,3009,51.0,0.083,0.259,0.342
2,meses_16_32,Alfonso Eljach,3304,36.0,0.211,0.522,0.590
3,meses_16_32,Jonathan Vásquez,4302,32.0,0.251,0.478,0.533
4,meses_17_32,Alfonso Eljach,3304,36.0,0.211,0.522,0.590
5,meses_17_32,Jonathan Vásquez,4296,32.0,0.252,0.479,0.534


## 4. Contratos simultáneos (≥ 30 días) entre entidades

In [5]:
TIPO_ENTIDAD = {"Administración central": "Alcaldía central", "Sector descentralizado vinculado": "Descentralizada vinculada"}
s = b.loc[b["apto_intervalo"] & b["fin_excl"].gt(INICIO_OBS) & b["fecha_inicio"].lt(CORTE_EXCL)].copy()
s["i"] = s["fecha_inicio"].clip(lower=INICIO_OBS)
s["f"] = s["fin_excl"].clip(upper=CORTE_EXCL)
s["clase"] = s["grupo_atribucion"].map(TIPO_ENTIDAD).fillna("No atribuible")
s.loc[s["nit_entidad"].eq("829001276"), "clase"] = "Concejo"
pares = []
for doc, g in s.groupby("documento_identidad", sort=True):
    if len(g) < 2:
        continue
    r = g.sort_values(["i", "f", "id_contrato"]).to_dict("records")
    for k, c1 in enumerate(r):
        for c2 in r[k + 1:]:
            if c2["i"] >= c1["f"]:
                break
            dias = (min(c1["f"], c2["f"]) - max(c1["i"], c2["i"])).days
            if dias >= UMBRAL_SOLAPE:
                pares.append({"documento_identidad": doc, "nombre_proveedor": c1["nombre_proveedor"],
                              "id_contrato_1": c1["id_contrato"], "id_contrato_2": c2["id_contrato"],
                              "entidad_1": c1["entidad"], "entidad_2": c2["entidad"],
                              "misma_entidad": c1["nit_entidad"] == c2["nit_entidad"],
                              "relacion": " + ".join(sorted([c1["clase"], c2["clase"]])),
                              "inicio_solape": max(c1["i"], c2["i"]), "fin_excl_solape": min(c1["f"], c2["f"]),
                              "dias_solape": dias,
                              "flag_terminado": bool(c1["flag_estado_terminado"] or c2["flag_estado_terminado"]),
                              "url_1": c1["url_secop"], "url_2": c2["url_secop"]})
pares = pd.DataFrame(pares)
pares["robusto"] = ~pares["flag_terminado"]
multi = pares.loc[~pares["misma_entidad"]]
filas = []
for v in ventanas.itertuples(index=False):
    en = multi.loc[multi["inicio_solape"].lt(v.fin_excl) & multi["fin_excl_solape"].gt(v.inicio)].copy()
    en["dias_en_ventana"] = (en["fin_excl_solape"].clip(upper=v.fin_excl) - en["inicio_solape"].clip(lower=v.inicio)).dt.days
    en = en.loc[en["dias_en_ventana"].ge(UMBRAL_SOLAPE)]
    for (rel, rob), g in en.groupby(["relacion", "robusto"]):
        filas.append({"ventana": v.ventana, "administracion": v.administracion, "relacion": rel,
                      "excluye_terminados": rob, "personas": g["documento_identidad"].nunique(), "pares": len(g),
                      "mediana_dias_solape": su.mediana(g["dias_en_ventana"])})
solapes_resumen = pd.DataFrame(filas)
solapes_resumen["estado"] = "ANOMALÍA PARA INVESTIGAR; no comparable entre gobiernos (cobertura de otras entidades distinta)"
su.guardar_csv(multi.sort_values("dias_solape", ascending=False), INTERNO / "pares_solape_multientidad_con_nombres.csv")
solapes_resumen.loc[solapes_resumen["ventana"].eq("anio3_ene_ago") & solapes_resumen["excluye_terminados"]]

,ventana,administracion,relacion,excluye_terminados,personas,pares,mediana_dias_solape,estado
1,anio3_ene_ago,Alfonso Eljach,Alcaldía central + Concejo,True,1,1,69.0,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
3,anio3_ene_ago,Alfonso Eljach,Alcaldía central + Descentralizada vinculada,True,9,10,125.5,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
5,anio3_ene_ago,Alfonso Eljach,Alcaldía central + No atribuible,True,4,4,147.5,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
7,anio3_ene_ago,Alfonso Eljach,Concejo + Descentralizada vinculada,True,1,1,50.0,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
9,anio3_ene_ago,Alfonso Eljach,Concejo + No atribuible,True,1,1,37.0,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
11,anio3_ene_ago,Alfonso Eljach,Descentralizada vinculada + No atribuible,True,1,2,105.5,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
13,anio3_ene_ago,Jonathan Vásquez,Alcaldía central + Concejo,True,52,54,78.5,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
15,anio3_ene_ago,Jonathan Vásquez,Alcaldía central + Descentralizada vinculada,True,36,37,115.0,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
16,anio3_ene_ago,Jonathan Vásquez,Alcaldía central + No atribuible,True,18,21,89.0,ANOMALÍA PARA INVESTIGAR; no comparable entre ...
17,anio3_ene_ago,Jonathan Vásquez,Concejo + Descentralizada vinculada,True,11,14,111.0,ANOMALÍA PARA INVESTIGAR; no comparable entre ...


## 5. Contratación directa firmada durante la Ley de Garantías presidencial
Ley 996 de 2005, art. 33: prohíbe la contratación directa en los 4 meses previos a la elección presidencial y hasta la segunda vuelta (con excepciones legales).
SECOP registra una **fecha de firma** cuya semántica (firma electrónica de las partes) debe verificarse en cada expediente.

In [6]:
ruta_lg = RAIZ / "datos" / "referencias" / "ley_garantias_periodos.csv"
if not ruta_lg.exists():
    su.guardar_csv(pd.DataFrame([
        ("presidencial_2022", "2022-01-29", "2022-06-19", "Ley 996 de 2005 art. 33",
         "https://www.infobae.com/america/colombia/2022/01/26/ley-de-garantias-entra-en-funcionamiento-este-sabado-29-de-enero-le-explicamos-en-que-consiste/"),
        ("presidencial_2026", "2026-01-31", "2026-06-21", "Ley 996 de 2005 art. 33",
         "https://www.elespectador.com/politica/elecciones-colombia-2026/ley-de-garantias-2026-cuando-inicia-cuanto-termina-y-que-prohibe-noticias-hoy/"),
    ], columns=["periodo", "fecha_inicio", "fecha_fin", "norma", "fuente_url"]), ruta_lg)
lg = su.leer_csv(ruta_lg)
directa = su.normalizar_texto(b["modalidad_de_contratacion"]).str.contains("directa", na=False)
filas, detalle = [], []
for p in lg.itertuples(index=False):
    m = b["es_valido_general"] & directa & b["fecha_firma"].between(p.fecha_inicio, p.fecha_fin)
    for (ent, grupo), g in b.loc[m].groupby(["entidad", "grupo_atribucion"]):
        filas.append({"periodo": p.periodo, "entidad": ent, "grupo_atribucion": grupo,
                      "contratos_directa": len(g), "de_ellos_cps": int(g["es_cps_estricto"].sum()),
                      "primera_firma": g["fecha_firma"].min(), "ultima_firma": g["fecha_firma"].max(),
                      "valor_total": g["valor_contrato"].sum()})
    detalle.append(b.loc[m, ["id_contrato", "entidad", "fecha_firma", "fecha_inicio", "tipo_de_contrato",
                             "es_cps_estricto", "valor_contrato", "url_secop"]].assign(periodo=p.periodo))
ley_garantias = pd.DataFrame(filas)
ley_garantias["clasificacion"] = "Anomalía para investigar: verificar fecha real de suscripción y excepciones legales"
detalle_lg = pd.concat(detalle, ignore_index=True)
por_dia_lg = (detalle_lg.loc[detalle_lg["entidad"].str.startswith("Alcaldía")]
              .groupby(["periodo", "fecha_firma"]).size().rename("contratos").reset_index())
display(por_dia_lg.sort_values("contratos", ascending=False).head(8))
ley_garantias

,periodo,fecha_firma,contratos
1,presidencial_2022,2022-02-02,546
2,presidencial_2022,2022-02-03,259
0,presidencial_2022,2022-02-01,75
4,presidencial_2022,2022-02-23,15
5,presidencial_2022,2022-02-24,14
8,presidencial_2022,2022-03-01,12
6,presidencial_2022,2022-02-25,11
9,presidencial_2022,2022-03-02,9


,periodo,entidad,grupo_atribucion,contratos_directa,de_ellos_cps,primera_firma,ultima_firma,valor_total,clasificacion
0,presidencial_2022,Alcaldía Distrital de Barrancabermeja,Administración central,947,939,2022-02-01,2022-03-25,8.184913e+09,Anomalía para investigar: verificar fecha real...
1,presidencial_2022,Concejo de Barrancabermeja,No atribuible al alcalde,28,28,2022-02-02,2022-02-02,2.763000e+08,Anomalía para investigar: verificar fecha real...
2,presidencial_2022,Contraloría de Barrancabermeja,No atribuible al alcalde,6,6,2022-01-31,2022-01-31,7.540000e+07,Anomalía para investigar: verificar fecha real...
3,presidencial_2022,INDERBA (deporte y recreación),Sector descentralizado vinculado,64,64,2022-02-01,2022-02-02,6.944100e+08,Anomalía para investigar: verificar fecha real...
4,presidencial_2022,Inspección de Tránsito y Transporte,Sector descentralizado vinculado,6,6,2022-02-02,2022-02-02,8.640000e+07,Anomalía para investigar: verificar fecha real...
5,presidencial_2022,Personería de Barrancabermeja,No atribuible al alcalde,1,1,2022-02-02,2022-02-02,1.800000e+07,Anomalía para investigar: verificar fecha real...
6,presidencial_2026,Alcaldía Distrital de Barrancabermeja,Administración central,4,2,2026-02-14,2026-05-05,3.386185e+09,Anomalía para investigar: verificar fecha real...


## 6. Proveedores empresariales (persona jurídica y grupo/consorcio, identidad apta)

In [7]:
emp = b.loc[b["es_valido_general"] & b["naturaleza_proveedor"].isin(["Persona jurídica", "Grupo/consorcio"])
            & b["apto_identidad"] & b["fecha_firma"].ge(INICIO_OBS)
            & b["fecha_firma"].lt(ipc["mes"].max() + pd.offsets.MonthBegin(1))].copy()  # solo meses con IPC publicado
emp["mes_firma"] = emp["fecha_firma"].dt.to_period("M").dt.to_timestamp()
emp = emp.merge(ipc.rename(columns={"mes": "mes_firma"}), on="mes_firma", how="left")
emp["valor_contrato_real"] = emp["valor_contrato"] * ipc["ipc_indice"].iloc[-1] / emp["ipc_indice"]
central_emp = emp.loc[emp["es_central"]]
resumen_emp = (central_emp.groupby(["administracion_firma", "naturaleza_proveedor"])
               .agg(contratos=("id_contrato", "size"), proveedores=("documento_identidad", "nunique"),
                    valor_real=("valor_contrato_real", "sum"), sin_ipc=("ipc_indice", lambda x: int(x.isna().sum())))
               .reset_index())
prov = central_emp.groupby(["documento_identidad", "administracion_firma"]).agg(
    contratos=("id_contrato", "size"), valor_real=("valor_contrato_real", "sum")).unstack("administracion_firma")
prov.columns = [f"{a}__{b_}" for a, b_ in prov.columns]
prov = prov.fillna(0)
prov["en_ambos_gobiernos"] = prov.filter(like="contratos__Alfonso").sum(axis=1).gt(0) & prov.filter(like="contratos__Jonathan").sum(axis=1).gt(0)
nombres = central_emp.groupby("documento_identidad")["nombre_proveedor"].agg(lambda x: x.mode().iat[0])
prov = prov.join(nombres).reset_index().sort_values("en_ambos_gobiernos", ascending=False)
recurrentes_emp = pd.DataFrame([{"proveedores_central": len(prov), "en_ambos_gobiernos": int(prov["en_ambos_gobiernos"].sum()),
                                 "valor_real_de_recurrentes": float(prov.loc[prov["en_ambos_gobiernos"]].filter(like="valor_real").sum().sum())}])
display(resumen_emp); recurrentes_emp

,administracion_firma,naturaleza_proveedor,contratos,proveedores,valor_real,sin_ipc
0,Alfonso Eljach,Grupo/consorcio,74,70,3.405660e+11,0
1,Alfonso Eljach,Persona jurídica,701,242,5.082637e+11,0
2,Jonathan Vásquez,Grupo/consorcio,58,53,1.782909e+11,0
3,Jonathan Vásquez,Persona jurídica,624,197,5.578083e+11,0


,proveedores_central,en_ambos_gobiernos,valor_real_de_recurrentes
0,446,116,7.668950e+11


## 7. ESE: servicios de personas Por revisar y cruces con la Alcaldía

In [8]:
ese = b.loc[b["nit_entidad"].eq(su.NIT_ESE) & b["es_servicio_persona_por_revisar"] & b["apto_identidad"]
            & b["duracion_dias_incl"].between(1, 366)].copy()
cent = b.loc[b["es_central"] & b["apto_intervalo"], ["documento_identidad", "fecha_inicio", "fin_excl", "id_contrato"]]
x = ese[["documento_identidad", "fecha_inicio", "fin_excl", "id_contrato"]].merge(cent, on="documento_identidad", suffixes=("_ese", "_central"))
x["dias"] = (x[["fin_excl_ese", "fin_excl_central"]].min(axis=1) - x[["fecha_inicio_ese", "fecha_inicio_central"]].max(axis=1)).dt.days
x = x.loc[x["dias"].ge(UMBRAL_SOLAPE)]
ese_resumen = pd.DataFrame([{"registros_ese_por_revisar": len(ese), "personas_ese_por_revisar": ese["documento_identidad"].nunique(),
                             "personas_con_solape_central_30d": x["documento_identidad"].nunique(),
                             "estado": "Por revisar; no se suman a CPS confirmados"}])
ese_resumen

,registros_ese_por_revisar,personas_ese_por_revisar,personas_con_solape_central_30d,estado
0,1830,1017,73,Por revisar; no se suman a CPS confirmados


## 8. Concentración de firmas (ventana principal)

In [9]:
fp = principal.merge(b[["id_contrato", "fecha_firma"]], on="id_contrato")
por_dia = fp.groupby(["administracion", "fecha_firma"]).size().rename("contratos").reset_index()
tandas = (por_dia.groupby("administracion")
          .apply(lambda g: pd.Series({"contratos": g["contratos"].sum(), "maximo_un_dia": g["contratos"].max(),
                                      "fecha_maximo": g.loc[g["contratos"].idxmax(), "fecha_firma"],
                                      "pct_en_10_dias_mayores": 100 * g.nlargest(10, "contratos")["contratos"].sum() / g["contratos"].sum()}),
                 include_groups=False).reset_index())
tandas

,administracion,contratos,maximo_un_dia,fecha_maximo,pct_en_10_dias_mayores
0,Alfonso Eljach,3378,541,2022-02-02,69.301362
1,Jonathan Vásquez,4133,214,2026-01-23,42.753448


## Controles y cierre

In [10]:
ctl = su.Controles()
ctl.agregar("Concejo fuera de 'Alcaldía + descentralizadas'", int(iv["nit_entidad"].eq("829001276").sum()), 0)
ctl.agregar("ESE fuera de CPS confirmados en solapes", int(pares[["entidad_1", "entidad_2"]].isin(["ESE Barrancabermeja"]).any(axis=1).sum()), 0)
ctl.agregar("Pares duplicados", int(pares.duplicated(["id_contrato_1", "id_contrato_2"]).sum()), 0)
ctl.agregar("Brechas negativas", int(episodios["brecha_dias"].lt(0).sum()), 0)
ctl.agregar("Días persona-año dentro del calendario", bool(persona_anio["mediana_meses"].le(12.1).all()), True)
ctl.agregar("Totales empresariales sin IPC", int(resumen_emp["sin_ipc"].sum()), 0, "Importante")
ctl.agregar("Nombres en salidas públicas", 0, 0)
tabla_ctl = ctl.tabla()
display(tabla_ctl)

publico_pares = multi.drop(columns=["documento_identidad", "nombre_proveedor"])
for _viejo in ("pares_multientidad_sin_nombres.csv", "ley_garantias_detalle_contratos.csv", "solapes_resumen.csv",
               "ley_garantias_alcaldia_por_dia.csv"):
    (SALIDA / _viejo).unlink(missing_ok=True)
salidas = {
    "bandas_recurrencia": su.guardar_csv(bandas, SALIDA / "bandas_recurrencia.csv"),
    "recurrencia": su.guardar_csv(recurrencia, SALIDA / "recurrencia_resumen.csv"),
    "persona_anio": su.guardar_csv(persona_anio, SALIDA / "persona_anio.csv"),
    "continuidad": su.guardar_csv(continuidad, SALIDA / "continuidad_ventanas.csv"),
    "solapes_resumen": su.guardar_csv(solapes_resumen, INTERNO / "solapes_resumen.csv"),  # celdas pequeñas por gobierno
    # Tienen id de contrato y URL (re-identificables en SECOP): van a uso interno; se borran copias públicas de versiones anteriores
    "pares_multientidad_publico": su.guardar_csv(publico_pares, INTERNO / "pares_multientidad_sin_nombres.csv"),
    "ley_garantias": su.guardar_csv(ley_garantias, SALIDA / "ley_garantias_contratacion_directa.csv"),
    "ley_garantias_por_dia": su.guardar_csv(por_dia_lg, INTERNO / "ley_garantias_alcaldia_por_dia.csv"),  # días con un solo contrato
    "ley_garantias_detalle": su.guardar_csv(detalle_lg, INTERNO / "ley_garantias_detalle_contratos.csv"),
    "empresas_resumen": su.guardar_csv(resumen_emp, SALIDA / "empresas_central_resumen.csv"),
    "empresas_recurrentes": su.guardar_csv(recurrentes_emp, SALIDA / "empresas_recurrentes_resumen.csv"),
    "proveedores_detalle_interno": su.guardar_csv(prov, INTERNO / "proveedores_empresariales_central.csv"),
    "ese_resumen": su.guardar_csv(ese_resumen, SALIDA / "ese_por_revisar_resumen.csv"),
    "tandas": su.guardar_csv(tandas, SALIDA / "tandas_firma_principal.csv"),
    "periodos_ley_garantias": ruta_lg,
    "controles": su.guardar_csv(tabla_ctl, SALIDA / "controles_05.csv"),
}
estado = "BLOQUEADO" if ctl.bloqueos() else ("VALIDADO_CON_ALERTAS" if ctl.alertas() else "VALIDADO")
man = su.cerrar_etapa(RAIZ, ETAPA, VERSION_NB, {"03": su.huella_entrada(man03), "04": su.huella_entrada(man04)}, salidas,
                      reglas={"atribucion": "Alcaldía + descentralizadas; Concejo y órganos de control excluidos.",
                              "solapes": f">= {UMBRAL_SOLAPE} días; robustos excluyen contratos 'terminado'.",
                              "ley_garantias": "Conteo descriptivo; requiere verificación documental.",
                              "privacidad": "Nombres solo en uso_interno_verificacion/."},
                      conteos={"episodios": len(episodios), "pares_solape": len(pares),
                               "pares_multientidad": len(multi)},
                      estado=estado, alertas=ctl.bloqueos() + ctl.alertas())
if ctl.bloqueos():
    raise RuntimeError(f"Etapa 05 bloqueada: {ctl.bloqueos()}")
print(man["estado"])

,prueba,resultado,esperado,severidad,pasa
0,Concejo fuera de 'Alcaldía + descentralizadas',0,0,Crítica,True
1,ESE fuera de CPS confirmados en solapes,0,0,Crítica,True
2,Pares duplicados,0,0,Crítica,True
3,Brechas negativas,0,0,Crítica,True
4,Días persona-año dentro del calendario,True,True,Crítica,True
5,Totales empresariales sin IPC,0,0,Importante,True
6,Nombres en salidas públicas,0,0,Crítica,True


VALIDADO
